# RoboPaintBench: A Pelican Riding a Bicycle

Replay one of the 11 published text-goal painting sessions in the native Linux simulator.

> Paint a pelican riding a bicycle.

**GPU rendering by default · No model API key · No reference image**

The model observed two fixed workspace cameras. A human-only pen-follow camera may appear in motion recordings, but was not an extra model input. The completed collection has 33 attempts and 603 actions; it has no artistic ranking.

This notebook selects Pelican records and the Pelican configuration. It does not call the selected model. Its new case selector and source bundle have offline tests; a hosted Pelican Colab GPU run has not yet been validated. The Golden Gate notebook and its dated hosted validation remain separate.


## Quick start

1. Open this notebook in Colab and sign in. If the GitHub repository requires access, upload the local `.ipynb`; its source bundle is embedded.
2. Choose **Runtime → Change runtime type → Python 3 / T4 GPU** and connect.
3. Keep embedded source, GPU rendering, replay mode, startup enabled and optional MP4 recording off.
4. To select another source model, change **RECORD_MODEL** in step 5 before the first **Run all**. The default is Sol: 20 / 14 / 14 actions, 48 total. Every source record contains its model's three attempts.
5. **Run all**, then open the temporary native Colab viewer URL printed in step 7 using the same Google account. Wait for connected cameras, the text goal and advancing action history.
6. Inspect the actual Unity NVIDIA renderer and submitted canvases in step 9. Save outputs, then stop this session in step 10 before preparing another.

`human` mode is an optional manual session on the same text-only target and budget. It produces a separate demonstration, not a benchmark model result. Neither mode requests a model API key or constructs a model API agent.

The 11 public records are pinned by file size and SHA-256 to HF commit `b16fb52e866ce6a33c52e287f04bfe4a77964453`. Use complete results records rather than trajectories: six first-attempt trajectory exports contain only the final 24 steps. Colab timing and cross-machine pixel equality are not guaranteed.

## Set up


In [ ]:
# @title 1. Source and session { display-mode: "form" }
# @markdown Verify and unpack the embedded scripts. No separate upload is needed. Use **Show code** for advanced source and local-path settings.
SOURCE_MODE = "embedded"
SOURCE_ARCHIVE = ""
SOURCE_ARCHIVE_SHA256 = '0d9ec17df8c67330846deffcca30ebc7991bbe8a38f5a1b1157786123b042795'
LOCAL_CHECKOUT = ""
BENCHMARK_REF = "main"
EXISTING_PYTHON = ""
EXISTING_PLAYER_DIRECTORY = ""
# The single-file default embeds the verified minimal source archive below.
# Advanced users may select upload/github or supply LOCAL_CHECKOUT.
# SOURCE_ARCHIVE and its trusted hash apply to the optional upload route.

from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, io, json, os, re, stat, subprocess, sys, uuid, zipfile

if sys.platform != "linux" or not Path("/content").is_dir():
    raise RuntimeError("Use Colab's Linux runtime. The helper also supports a local Linux machine.")
SESSION = Path("/content") / ("pelican-record-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
SESSION.mkdir(exist_ok=False)
os.environ.update(AGENTARK_AUTO_LOAD_DOTENV="0", AUTO_LOAD_DOTENV="0", GIT_TERMINAL_PROMPT="0")

SOURCE_FILES = ('scripts/colab_record.py', 'scripts/colab_graphics.py', 'scripts/install_runtime.py', 'scripts/prepare_run.py', 'scripts/run_eval.py', 'scripts/package_colab_source.py', 'benchmarks/versions.json', 'benchmarks/task107_linux.task-v1.files.json', 'benchmarks/golden_gate/eval.yaml', 'benchmarks/golden_gate/painting_goal.txt', 'benchmarks/results/golden_gate.json', 'benchmarks/pelican/eval.yaml', 'benchmarks/pelican/painting_goal.txt', 'benchmarks/results/pelican.json')

# Verified source only; no records, media, secrets or runtime binaries.
EMBEDDED_SOURCE_ZIP_B64 = (
    '...'
)


def unpack_source_bundle(archive, destination, expected_sha256):
    # Bootstrap verification is independent of the code inside the archive.
    if not re.fullmatch(r"[0-9a-fA-F]{64}", expected_sha256):
        raise ValueError("Set SOURCE_ARCHIVE_SHA256 from the trusted packaging report.")
    if archive.stat().st_size > 2 * 1024 * 1024:
        raise ValueError("Source ZIP exceeds the 2 MiB limit.")
    archive_bytes = archive.read_bytes()
    actual_sha = hashlib.sha256(archive_bytes).hexdigest()
    if actual_sha != expected_sha256.lower():
        raise ValueError("Source ZIP SHA-256 does not match the trusted packaging report.")
    allowed = set(SOURCE_FILES) | {"SOURCE_MANIFEST.json"}
    payloads = {}
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as bundle:
        infos = bundle.infolist()
        names = [entry.filename for entry in infos]
        if (len(names) != len(allowed) or set(names) != allowed
                or len({name.casefold() for name in names}) != len(names)):
            raise ValueError("Source ZIP must contain exactly the curated source files and manifest.")
        for entry in infos:
            mode = entry.external_attr >> 16
            maximum = 16 * 1024 if entry.filename == "SOURCE_MANIFEST.json" else 512 * 1024
            if (entry.orig_filename != entry.filename or entry.is_dir()
                    or stat.S_IFMT(mode) != stat.S_IFREG or mode != 0o100644 or entry.external_attr & 0x400
                    or entry.flag_bits & 1 or entry.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)
                    or not 0 <= entry.file_size <= maximum):
                raise ValueError("Unsupported source ZIP member type, mode or size.")
            payloads[entry.filename] = bundle.read(entry)
    def unique_keys(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError("Duplicate source manifest JSON key.")
            result[key] = value
        return result
    manifest = json.loads(payloads.pop("SOURCE_MANIFEST.json").decode("utf-8"), object_pairs_hook=unique_keys)
    if not isinstance(manifest, dict) or set(manifest) != {"schema", "files"} or manifest["schema"] != "robopaint.colab-source.v1":
        raise ValueError("Unsupported source manifest.")
    entries = manifest["files"]
    if (not isinstance(entries, list) or not all(isinstance(item, dict) for item in entries)
            or [item.get("path") for item in entries] != sorted(SOURCE_FILES)):
        raise ValueError("Source manifest file list differs from the fixed allowlist.")
    total = 0
    for item in entries:
        if (set(item) != {"path", "bytes", "sha256"} or type(item["bytes"]) is not int
                or not 0 <= item["bytes"] <= 512 * 1024
                or not re.fullmatch(r"[0-9a-f]{64}", str(item["sha256"]))):
            raise ValueError("Invalid per-file source manifest entry.")
        content = payloads[item["path"]]
        if len(content) != item["bytes"] or hashlib.sha256(content).hexdigest() != item["sha256"]:
            raise ValueError("Source ZIP member differs from its size/SHA-256 manifest.")
        total += len(content)
    if total > 1024 * 1024:
        raise ValueError("Source payload exceeds 1 MiB.")
    if destination.exists() or destination.is_symlink():
        raise FileExistsError("Source destination already exists; choose a fresh session.")
    destination.mkdir(exist_ok=False)
    for name, content in payloads.items():
        target = destination / name  # Every name matched the exact ASCII allowlist above.
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as stream:
            stream.write(content)
    (destination / "SOURCE_MANIFEST.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return {"archive_sha256": actual_sha, "files": len(entries), "payload_bytes": total}


def run(command, **kwargs):
    command = [str(item) for item in command]
    if command and Path(command[0]).name == "apt-get":
        child_env = dict(kwargs.get("env", os.environ))
        child_env["DEBIAN_FRONTEND"] = "noninteractive"
        kwargs["env"] = child_env
    # These arguments contain public versions/paths, never credentials.
    print("+", " ".join(command), flush=True)
    return subprocess.run(command, check=True, **kwargs)

SOURCE_RECEIPT = {"mode": "local_checkout" if LOCAL_CHECKOUT.strip() else SOURCE_MODE}
if LOCAL_CHECKOUT.strip():
    BENCH = Path(LOCAL_CHECKOUT).expanduser().resolve(strict=True)
elif SOURCE_MODE == "embedded":
    archive = SESSION / "embedded-colab-source.zip"
    with archive.open("xb") as stream:
        stream.write(base64.b64decode(EMBEDDED_SOURCE_ZIP_B64, validate=True))
    BENCH = SESSION / "benchmark"
    SOURCE_RECEIPT.update(unpack_source_bundle(archive, BENCH, SOURCE_ARCHIVE_SHA256))
elif SOURCE_MODE == "upload":
    if not re.fullmatch(r"[0-9a-fA-F]{64}", SOURCE_ARCHIVE_SHA256):
        raise ValueError("Enter the curated source ZIP's SHA-256 from the trusted packaging report.")
    if SOURCE_ARCHIVE.strip():
        archive = Path(SOURCE_ARCHIVE).expanduser().resolve(strict=True)
    else:
        from google.colab import files
        uploads = SESSION / "source-upload"
        uploads.mkdir(exist_ok=False)
        previous = Path.cwd()
        try:
            os.chdir(uploads)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one curated source ZIP.")
        name = next(iter(uploaded))
        if Path(name).name != name:
            raise ValueError("Use a simple source ZIP basename.")
        archive = uploads / name
    BENCH = SESSION / "benchmark"
    SOURCE_RECEIPT.update(unpack_source_bundle(archive, BENCH, SOURCE_ARCHIVE_SHA256))
elif SOURCE_MODE == "github":
    BENCH = SESSION / "benchmark"
    run(["git", "clone", "--filter=blob:none", "https://github.com/P90-RushB/robopaint-bench.git", BENCH])
    run(["git", "-C", BENCH, "checkout", "--detach", BENCHMARK_REF])
    SOURCE_RECEIPT["commit"] = subprocess.check_output(["git", "-C", str(BENCH), "rev-parse", "HEAD"], text=True).strip()
else:
    raise ValueError("Choose embedded, upload or github source mode.")
LOCK = json.loads((BENCH / "benchmarks/versions.json").read_text("utf-8"))
HELPER = BENCH / "scripts/colab_record.py"
if not HELPER.is_file():
    raise RuntimeError("Use a checkout containing the recording notebook/helper.")
print("Session:", SESSION)
print("Source receipt:", SOURCE_RECEIPT)
print("AgentArk source pin:", LOCK["agentark"]["commit"])


In [ ]:
# @title 2. Install dependencies { display-mode: "form" }
# @markdown Install dependencies in a separate **Python 3.10.12** environment. Wait for installation to finish; the Colab kernel stays unchanged.
if EXISTING_PYTHON.strip():
    PYTHON = Path(EXISTING_PYTHON).expanduser().resolve(strict=True)
else:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git", "xvfb", "xauth", "mesa-utils", "xserver-xorg-core", "libgl1", "libgl1-mesa-dri", "libglib2.0-0", "libxrandr2", "libxcursor1", "libxi6", "libxinerama1", "libpulse0", "libnss3", "ffmpeg"])
    run([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.7"])
    run([sys.executable, "-m", "uv", "python", "install", "3.10.12", "--no-bin"])
    VENV = SESSION / "python-env"
    run([sys.executable, "-m", "uv", "venv", "--python", "3.10.12", "--seed", VENV])
    PYTHON = VENV / "bin/python"
    AGENTARK = SESSION / "AgentArk"
    run(["git", "clone", "--filter=blob:none", "https://github.com/P90-RushB/AgentArk.git", AGENTARK])
    run(["git", "-C", AGENTARK, "checkout", "--detach", LOCK["agentark"]["commit"]])
    run([PYTHON, "-m", "pip", "install", "-q", "-e", AGENTARK, "huggingface_hub==0.36.0"])
run([PYTHON, "-c", "import sys; assert sys.version_info[:3] == (3,10,12); import agent_ark; print('Python:', sys.version.split()[0]); print('AgentArk:', agent_ark.__file__)"])
print("The package's Python constraint is >=3.10,<3.10.13. No model SDK/login is needed.")

In [ ]:
# @title 3. Download the simulator { display-mode: "form" }
# @markdown Download and verify the simulator. No Unity Editor, model API key or Drive mount is needed. Default public downloads require no login.
HF_AUTH_REQUIRED = False
if HF_AUTH_REQUIRED and not os.environ.get("HF_TOKEN"):
    import getpass
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    token = token or getpass.getpass("HF_TOKEN (hidden): ").strip()
    if not token:
        raise RuntimeError("Dataset access was requested but no token was supplied.")
    os.environ["HF_TOKEN"] = token
    del token
if EXISTING_PLAYER_DIRECTORY.strip():
    PLAYER = Path(EXISTING_PLAYER_DIRECTORY).expanduser().resolve(strict=True)
else:
    PLAYER = SESSION / "player"
    run([PYTHON, BENCH / "scripts/install_runtime.py", "--platform", "linux", "--destination", PLAYER])
MODS = PLAYER / "AgentArk_Data/Resources/Mods"
STORE = MODS / "all_tasks"
os.environ.update(AGENTARK_ENV_PATH=str(PLAYER / "AgentArk.x86_64"), AGENTARK_MOD_PATH=str(MODS), AGENTARK_TASK_STORE_PATH=str(STORE))
print("Runtime selected. The next preparation checks its pinned task files and inputs.")

## Render and prepare

**GPU is the default.** Select a T4 GPU runtime before continuing. Step 4 verifies the NVIDIA display. CPU rendering is used only when you explicitly select `cpu`.

Step 5 selects replay or human mode and prepares a fresh run. Step 6 starts the simulator.


In [ ]:
# @title 4. GPU rendering or explicit CPU fallback { display-mode: "form" }
# @markdown Use NVIDIA GPU rendering by default. Select a T4 GPU runtime first. Choose `cpu` only for the slower software-rendering fallback.
RENDERER = "gpu"  # @param ["gpu", "cpu"]
GRAPHICS_ENV = {}
GRAPHICS_UNSET_ENV = []
if RENDERER == "gpu":
    if "GRAPHICS_DIR" in globals():
        if not (GRAPHICS_DIR / "graphics.json").is_file():
            raise RuntimeError("The previous display receipt is missing. Its directory identity is preserved; reconnect a fresh Colab runtime before setting up another display.")
        graphics = json.loads((GRAPHICS_DIR / "graphics.json").read_text("utf-8"))
        try:
            argv = (Path("/proc") / str(int(graphics["xorg_pid"])) / "cmdline").read_bytes().split(b"\0")
            argv = [part.decode("utf-8") for part in argv if part]
        except (OSError, ValueError, UnicodeError) as error:
            raise RuntimeError("The saved display identity cannot be verified. Its receipt is preserved; reconnect a fresh Colab runtime before setting up another display.") from error
        config_index = argv.index("-config") if "-config" in argv else -1
        if (not argv or Path(argv[0]).name != "Xorg" or graphics["display"] not in argv
                or config_index < 0 or config_index + 1 >= len(argv)
                or argv[config_index + 1] != str(GRAPHICS_DIR / "xorg.conf")):
            raise RuntimeError("The saved display process identity changed. No new display was started and the receipt was preserved.")
        verify_env = os.environ.copy()
        for key in graphics["unset_environment"]:
            verify_env.pop(key, None)
        verify_env.update(graphics["environment"])
        glx = subprocess.run(["glxinfo", "-B"], env=verify_env, capture_output=True, text=True, check=True)
        if ("OpenGL vendor string: NVIDIA Corporation" not in glx.stdout
                or "OpenGL renderer string:" not in glx.stdout
                or any(term in glx.stdout.lower() for term in ("llvmpipe", "softpipe", "swrast"))):
            raise RuntimeError("The owned display no longer passes NVIDIA GLX verification; no replacement was started.")
        print("Reusing this notebook's verified NVIDIA display:", graphics["display"], flush=True)
    else:
        GRAPHICS_DIR = SESSION / ("graphics-" + uuid.uuid4().hex[:8])
        run([PYTHON, BENCH / "scripts/colab_graphics.py", "--output-dir", GRAPHICS_DIR])
        graphics = json.loads((GRAPHICS_DIR / "graphics.json").read_text("utf-8"))
    GRAPHICS_ENV = graphics["environment"]
    GRAPHICS_UNSET_ENV = graphics["unset_environment"]
    RENDER_MODE, GRAPHICS_API = "existing-display", "glcore33"
    print("GPU display setup evidence:", GRAPHICS_DIR)
else:
    RENDER_MODE, GRAPHICS_API = "xvfb-software", "glcore33"
print("Requested renderer:", RENDERER, "API:", GRAPHICS_API)
print("Actual Unity renderer will be recorded in this run's player.log.")

In [ ]:
# @title 5. Choose replay or human mode { display-mode: "form" }
# @markdown `replay` follows the selected public Pelican record automatically; source-model selection never calls that model. `human` waits for actions in the viewer. MP4 recording is off; live viewing and action records still work.
if "session_process" in globals() and session_process.poll() is None:
    raise RuntimeError("The previous session/viewer is still running. Use the explicit cleanup cell before preparing another run.")
RECORD_MODEL = "sol"  # @param ["sol", "argon", "opus", "grok", "wonder", "deepseek", "qwen", "kimi", "mimo", "doubao", "astra"]
MODE = "replay"  # @param ["replay", "human"]
RECORD_MOTION_VIDEO = False  # @param {type:"boolean"}
VIEWER_PORT = 18181
BASE_PORT = 5005
RUN = SESSION / ("run-" + MODE + "-" + uuid.uuid4().hex[:8])
command = [PYTHON, HELPER, "prepare", "--case", "pelican", "--model", RECORD_MODEL, "--mode", MODE, "--run-dir", RUN, "--env-path", PLAYER / "AgentArk.x86_64", "--mod-path", MODS, "--task-store", STORE, "--render-mode", RENDER_MODE, "--graphics-api", GRAPHICS_API, "--viewer-port", str(VIEWER_PORT), "--base-port", str(BASE_PORT)]
if MODE == "replay":
    RECORD = SESSION / ("public-pelican-" + RECORD_MODEL + "-" + uuid.uuid4().hex[:8] + ".jsonl")
    run([PYTHON, HELPER, "download-record", "--case", "pelican", "--model", RECORD_MODEL, "--destination", RECORD])
    command.extend(["--record-path", RECORD])
run(command, cwd=BENCH)
CONFIG = RUN / "eval.yaml"
PREPARED_SELECTION = (RECORD_MODEL, MODE, RENDERER, RENDER_MODE, GRAPHICS_API, VIEWER_PORT, BASE_PORT)
(RUN / "python-packages.txt").write_text(subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True), encoding="utf-8")
(RUN / "source_receipt.json").write_text(json.dumps(SOURCE_RECEIPT, indent=2) + "\n", encoding="utf-8")
print("Prepared config:", CONFIG)
print("Unity/model/viewer have not been started. Neither mode needs an API model credential.")

## Watch

Run steps 6–7 to start the session and open AgentArk's native viewer. **Use the viewer for live drawing**, camera views and action progress.

If it is not ready yet, wait for startup, refresh the viewer and check the launch output or `session.log`. The link expires when this temporary runtime ends. Do not run the whole notebook again while a session is active.


In [ ]:
# @title 6. Start the session { display-mode: "form" }
# @markdown Leave startup enabled for **Run all**. Stop any active session with step 10 before preparing and launching another run.
START_SESSION = True  # @param {type:"boolean"}
if not START_SESSION:
    print("Prepared only. Toggle START_SESSION to launch the native simulator and viewer.")
else:
    if "session_process" in globals() and session_process.poll() is None:
        raise RuntimeError("The previous session/viewer is still running. Use the explicit cleanup cell before launching another run.")
    if PREPARED_SELECTION != (RECORD_MODEL, MODE, RENDERER, RENDER_MODE, GRAPHICS_API, VIEWER_PORT, BASE_PORT):
        raise RuntimeError("Selections changed after preparation. Prepare a fresh run.")
    if (RUN / "session_started.json").exists():
        raise RuntimeError("This run already started. Prepare a new run before launching again.")
    child_env = os.environ.copy()
    if RENDERER == "gpu":
        for key in GRAPHICS_UNSET_ENV:
            child_env.pop(key, None)
        child_env.update(GRAPHICS_ENV)
    # ML-Agents starts Unity in its own session. Translate TERM into a Python
    # interruption so the upstream finally/env.close path can close Unity.
    SIGNAL_BOOTSTRAP = 'import runpy, signal, sys\nfrom pathlib import Path\n\ndef requested_stop(signum, frame):\n    raise KeyboardInterrupt("Notebook requested session cleanup")\n\nsignal.signal(signal.SIGTERM, requested_stop)\nscript = sys.argv[1]\nsys.argv = sys.argv[1:]\nsys.path.insert(0, str(Path(script).resolve().parent))\nrunpy.run_path(script, run_name="__main__")\n'
    launch = [str(PYTHON), "-c", SIGNAL_BOOTSTRAP, str(HELPER), "run", "--config", str(CONFIG)]
    if RECORD_MOTION_VIDEO:
        import shutil
        ffmpeg = shutil.which("ffmpeg")
        if not ffmpeg:
            raise RuntimeError("Install ffmpeg before enabling motion recording.")
        launch.extend(["--record-video", "--ffmpeg", ffmpeg])
    with (RUN / "session.log").open("x", encoding="utf-8") as log:
        session_process = subprocess.Popen(launch, cwd=BENCH, env=child_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        session_process_group = session_process.pid
    print("Started PID:", session_process.pid, "mode:", MODE)
    print("Session log:", RUN / "session.log")
    print("Unity renderer log:", RUN / "player.log")

In [ ]:
# @title 7. Open the live viewer { display-mode: "form" }
# @markdown Click the native Colab URL below using the same Google account. Wait for **connected**, two camera views and the text-only painting goal; no reference image is appended.
from google.colab import output
print(output.eval_js(f"google.colab.kernel.proxyPort({int(VIEWER_PORT)})"))
print("Replay runs recorded actions automatically. Human mode waits for actions in the viewer.")
print("The viewer can remain available after a completed replay; check replay_results.jsonl/run_end, not just process exit.")

## Optional tools

**Human actions (step 8):** prefer the viewer's Human Action field. The notebook form is an optional alternative and sends nothing by default.

**Saved results and GPU check (step 9):** rerun after camera images appear to inspect the actual Unity renderer. A T4 run should report **NVIDIA / Tesla T4**; `llvmpipe` means CPU software rendering. Submitted canvases appear only after an attempt is explicitly submitted. A blank initialization is normal and is not a submitted work. Watch ongoing drawing in the **step 7 viewer**.

**Stop or switch modes (step 10):** download results you want to keep, then enable and run the stop form. To change modes, stop first and rerun steps **5 → 6 → 7**. When finished, disconnect and delete the runtime to release resources. Temporary files and viewer links disappear.


In [ ]:
# @title 8. Optional human action { display-mode: "form" }
# @markdown For **human mode** only. Prefer the viewer's **Human Action** field. This optional form sends nothing unless enabled.
SEND_ACTION = False  # @param {type:"boolean"}
ACTION = '<tool_call>{"name":"SubmitPainting","arguments":{}}</tool_call>'  # @param {type:"string"}
if SEND_ACTION:
    if MODE != "human":
        raise RuntimeError("Only human mode accepts these actions. Replay uses the saved record.")
    import urllib.request
    request = urllib.request.Request(f"http://127.0.0.1:{int(VIEWER_PORT)}/human/actions", data=json.dumps({"agent_id": None, "action": ACTION}).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(request, timeout=15) as response:
        print(response.read().decode("utf-8"))
else:
    print("No action submitted. Use the viewer or deliberately enable SEND_ACTION.")
print("A submit-only blank canvas is an interaction smoke, not evidence of drawing capability.")

In [ ]:
# @title 9. Optional renderer and submitted canvases { display-mode: "form" }
# @markdown Optional: inspect the actual Unity renderer and submitted canvases. For live drawing, open the **step 7 viewer**. This cell only refreshes saved results; it does not advance or reset the simulation. Full diagnostics are collapsed below.

def submitted_paintings(directory):
    import json, re
    from pathlib import Path
    directory = Path(directory)
    counts = {"metadata": 0, "staging": 0, "not_submitted": 0, "duplicates": 0, "incomplete": 0}
    selected, warnings = {}, []
    for path in sorted(directory.glob("**/metadata.json")):
        counts["metadata"] += 1
        if any(part.startswith(".staging_") for part in path.relative_to(directory).parts):
            counts["staging"] += 1
            continue
        try:
            metadata = json.loads(path.read_text("utf-8"))
            if metadata.get("terminal_status") != "submitted" or metadata.get("explicit_submission") is not True:
                counts["not_submitted"] += 1
                continue
            saved = metadata.get("saved_utc", "")
            guid = metadata.get("episode_guid", "")
            canvas = path.parent / "canvas.png"
            if (not isinstance(saved, str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{7}Z", saved)
                    or not isinstance(guid, str) or not guid.strip() or not canvas.is_file() or canvas.stat().st_size == 0):
                raise ValueError("Submission metadata, UTC save time or canvas file is not yet complete.")
            row = {"episode_guid": guid, "saved_utc": saved, "action_count": metadata.get("action_count"),
                   "submission_integrity": metadata.get("submission_integrity"), "canvas_path": str(canvas)}
            key = guid.strip().casefold()
            if key in selected:
                counts["duplicates"] += 1
            if key not in selected or saved > selected[key]["saved_utc"]:
                selected[key] = row
        except (OSError, ValueError, TypeError, AttributeError) as error:
            counts["incomplete"] += 1
            if len(warnings) < 5:
                warnings.append(str(path.relative_to(directory)) + ": " + str(error)[:160])
    return {"paintings": sorted(selected.values(), key=lambda item: (item["saved_utc"], item["episode_guid"]))[-3:],
            "unique_submitted": len(selected), "counts": counts, "warnings": warnings}


from IPython.display import HTML, Image, display
import html

try:
    inspection = subprocess.run([str(PYTHON), str(HELPER), "inspect", "--run-dir", str(RUN)], check=True, capture_output=True, text=True)
    evidence = json.loads(inspection.stdout)
    renderer_lines = [line for line in evidence.get("renderer_log_evidence", []) if re.search(r"Forcing GfxDevice|Renderer:|Vendor:|Version:", line)]
    print("Actual Unity renderer:")
    print("\n".join(renderer_lines) if renderer_lines else "Renderer log is not available yet. Wait for camera images, then rerun this cell.")
    display(HTML("<details><summary>Full inspection details</summary><pre>" + html.escape(json.dumps(evidence, ensure_ascii=False, indent=2)) + "</pre></details>"))
    if inspection.stderr:
        print(inspection.stderr, file=sys.stderr, end="")
except (subprocess.CalledProcessError, OSError, ValueError, TypeError, AttributeError) as error:
    print("Full renderer inspection is temporarily unavailable:", str(error)[:240])
    if getattr(error, "stderr", None):
        print(error.stderr, file=sys.stderr, end="")


print("\nSubmitted canvases (latest 3 by save time)")
submitted = submitted_paintings(RUN / "paintings")
print("Saved submissions:", submitted["unique_submitted"], "total.")
for warning in submitted["warnings"]:
    print("Not yet complete:", warning)
if not submitted["paintings"]:
    print("No submitted works yet. Watch live drawing in the step 7 viewer. The initial blank canvas is normal and is not a submitted work.")
for painting in submitted["paintings"]:
    print("Submitted · Saved (UTC):", painting["saved_utc"], "· Actions:", painting["action_count"],
          "· submission_integrity:", painting["submission_integrity"], "· GUID:", painting["episode_guid"])
    try:
        from PIL import Image as PILImage
        with PILImage.open(painting["canvas_path"]) as checked:
            checked.verify()
        display(Image(filename=painting["canvas_path"], width=640))
    except (OSError, ValueError, TypeError, ImportError) as error:
        print("This submitted canvas is not readable yet; rerun this cell shortly:", str(error)[:200])
if "session_process" in globals():
    print("Launcher process return code:", session_process.poll())


In [ ]:
# @title 10. Stop this session (default off) { display-mode: "form" }
# @markdown Enable only when finished watching or switching modes. Download wanted results first, then disconnect and delete the runtime when finished.
STOP_SESSION = False  # @param {type:"boolean"}

import signal


def stop_owned_session(process, owned_group, timeout_s=15):
    if owned_group != process.pid:
        raise RuntimeError("Session process ownership does not match; no signal was sent.")
    if process.poll() is not None:
        return "The tracked launcher already exited; no process was signaled."
    try:
        actual_group = os.getpgid(process.pid)
    except ProcessLookupError:
        return "The tracked launcher exited before cleanup; no process was signaled."
    if actual_group != owned_group:
        raise RuntimeError("Session process group changed; no signal was sent.")
    os.killpg(owned_group, signal.SIGTERM)
    try:
        process.wait(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        if process.poll() is None:
            try:
                actual_group = os.getpgid(process.pid)
            except ProcessLookupError:
                actual_group = None
            if actual_group == owned_group:
                os.killpg(owned_group, signal.SIGKILL)
            elif actual_group is not None:
                raise RuntimeError("Session process group changed during cleanup; escalation refused.")
        process.wait(timeout=5)
    return "Owned launcher group stopped. Check session.log for Unity shutdown and retained artifacts for completeness; after forced escalation, disconnect the runtime if child processes remain."


if STOP_SESSION:
    if "session_process" not in globals() or "session_process_group" not in globals():
        print("No session process owned by this notebook is available to stop.")
    else:
        print(stop_owned_session(session_process, session_process_group))
else:
    print("No cleanup requested. Set STOP_SESSION=true and run this cell to stop this notebook's current session/viewer.")
print("This cell does not stop NVIDIA Xorg or any unrelated process. Disconnecting the Colab runtime releases its display resources.")


## Appendix: reproducibility and evidence

The embedded source archive contains fourteen reviewed source/configuration/metadata files, with an independently embedded archive hash and exact member hashes checked before extraction. No runtime binaries, raw records, videos, HTTP request histories or credentials are bundled. Public records are downloaded separately using immutable HF paths, sizes and SHA-256 hashes.

AgentArk remains the evaluator and replay framework. This notebook calls the existing installer, `prepare_run.py` and `colab_record.py`; the latter delegates replay to `agent_ark.ark_eval.run_replay`. It does not implement its own action loop or model harness.

Pinned runtime: Linux env **1.0.5**, Task107 RobotPainting **task-v1**, with the commit/size/SHA locks in `benchmarks/versions.json`. No Unity Editor is needed. AgentArk runs in a separate Python 3.10.12 environment; this does not replace the Colab kernel.

Every replay keeps seed 107, three attempts, at most 96 actions each, fixed brush/paper, infinite pigment and 480 Hz physics. Keep `no_graphics: false`. NVIDIA GPU mode creates an Xorg display, verifies GLX, and saves Unity's actual renderer log; CPU/software fallback requires an explicit selection. A successful `nvidia-smi` alone does not certify Unity's renderer.

The full records contain 603 actions over 33 attempts. The first-attempt trajectories for Opus, Grok, Wonder, DeepSeek, Qwen and Doubao retain only the final 24 steps; use results records for complete replay. Model-specific counts and actual settings remain in `benchmarks/results/pelican.json`. In particular, Opus used high, MiMo had two temporary 65,536-token provider requests, and Astra used Codex SDK max without an API-equivalent hard cap. These settings do not create model calls during replay.

The separate published Pelican films show all three attempts at **4× simulated-time speed**, with Chinese/English captions and explicitly labeled reading pauses. This notebook runs native replay at the configured simulator speed; it does not claim its live viewer is the edited 4× film. Optional native videos use the task's recorder, not a desktop recorder. Match videos and canvases by episode GUID, check manifest complete/truncated/encoder status, fully decode MP4 and compare frame count to `frames.jsonl`. `inspect` alone does not perform video decoding.

This Pelican extension has offline checks, not a hosted GPU execution receipt. Golden Gate's existing hosted validation and public Drive notebook remain unchanged and must not be presented as Pelican evidence. A replay match verifies the action/termination contract; it is not a cross-platform pixel-equality guarantee or an artistic quality metric. No Pelican artistic ranking or numerical score is assigned.

References: [Pelican guide](https://github.com/P90-RushB/robopaint-bench/blob/main/docs/pelican-results.md), [AgentArk](https://github.com/P90-RushB/AgentArk), [HF dataset](https://huggingface.co/datasets/P90-RushB/AgentArk/tree/b16fb52e866ce6a33c52e287f04bfe4a77964453/artifacts/records/0100-0199/0107-RobotPainting/pelican).
